In [2]:

import os
import pickle
import math
from pprint import pprint

import numpy as np
import pandas as pd

# ADJUST THIS PATH
GDV_PKL_PATH = "/Users/juliahagen/Vision-Transformers/res/gdv_500_0/gdv.pkl"

assert os.path.exists(GDV_PKL_PATH), f"File not found: {GDV_PKL_PATH}"
print("Using pickle:", GDV_PKL_PATH)

Using pickle: /Users/juliahagen/Vision-Transformers/res/gdv_500_0/gdv.pkl


In [3]:
# Cell 2: safe unpickler copied from the Dash app

class RemappingUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)

def load_pickle_remap(path: str):
    with open(path, "rb") as f:
        return RemappingUnpickler(f).load()

obj = load_pickle_remap(GDV_PKL_PATH)

print("Top-level keys:")
print(sorted(obj.keys()))

Top-level keys:
['gdv_per_layer', 'gdv_per_layer_cosine', 'layer_data', 'meta', 'sorted_layers']


In [4]:
# Cell 3: inspect structure

sorted_layers = obj.get("sorted_layers", [])
layer_data = obj.get("layer_data", {})
gdv_per_layer = obj.get("gdv_per_layer", {})

print("Number of sorted_layers:", len(sorted_layers))
print("Number of layer_data entries:", len(layer_data))
print("Number of gdv_per_layer entries:", len(gdv_per_layer))

print("\nFirst 5 sorted_layers:")
print(sorted_layers[:5])

if sorted_layers:
    first_key = sorted_layers[0]
    print("\nFirst layer key:", first_key)
    print("Keys inside layer_data[first_key]:")
    print(sorted(layer_data[first_key].keys()))

Number of sorted_layers: 82
Number of layer_data entries: 82
Number of gdv_per_layer entries: 82

First 5 sorted_layers:
['vision_0_D1408', 'vision_1_D1408', 'vision_2_D1408', 'vision_3_D1408', 'vision_4_D1408']

First layer key: vision_0_D1408
Keys inside layer_data[first_key]:
['gdv_cosine', 'gdv_euclidean', 'inter_cosine', 'inter_euclidean', 'intra_cosine', 'intra_euclidean', 'labels', 'layer_num', 'modality', 'pooling', 'projections', 'samples', 'samples_used', 'texts', 'tsne_runs', 'umap_runs', 'width_D', 'x', 'y']


In [6]:
# Cell 4: compare gdv_per_layer vs layer_data[layer]["gdv_euclidean"]

rows = []

for layer_key in sorted_layers:
    gdv_global = gdv_per_layer.get(layer_key, None)

    ld = layer_data.get(layer_key, {})
    gdv_local = ld.get("gdv_euclidean", None)

    same = False
    diff = None

    if gdv_global is not None and gdv_local is not None:
        try:
            gdv_global_f = float(gdv_global)
            gdv_local_f = float(gdv_local)
            same = math.isclose(gdv_global_f, gdv_local_f, rel_tol=1e-12, abs_tol=1e-12)
            diff = gdv_global_f - gdv_local_f
        except Exception:
            pass

    rows.append({
        "layer_key": layer_key,
        "gdv_per_layer": gdv_global,
        "gdv_layer_data": gdv_local,
        "same_exactish": same,
        "difference": diff,
        "has_gdv_per_layer": gdv_global is not None,
        "has_gdv_layer_data": gdv_local is not None,
    })

df_compare = pd.DataFrame(rows)
df_compare

,layer_key,gdv_per_layer,gdv_layer_data,same_exactish,difference,has_gdv_per_layer,has_gdv_layer_data
0,vision_0_D1408,-0.332566,-0.332566,True,0.0,True,True
1,vision_1_D1408,-0.340178,-0.340178,True,0.0,True,True
2,vision_2_D1408,-0.344854,-0.344854,True,0.0,True,True
3,vision_3_D1408,-0.347748,-0.347748,True,0.0,True,True
4,vision_4_D1408,-0.349181,-0.349181,True,0.0,True,True
...,...,...,...,...,...,...,...
77,language_43_D5120,-0.460623,-0.460623,True,0.0,True,True
78,language_44_D5120,-0.464332,-0.464332,True,0.0,True,True
79,language_45_D5120,-0.464737,-0.464737,True,0.0,True,True
80,language_46_D5120,-0.457685,-0.457685,True,0.0,True,True


In [7]:
# Cell 5: summarize what is going on

print("Layers with gdv_per_layer present:", df_compare["has_gdv_per_layer"].sum())
print("Layers with gdv_layer_data present:", df_compare["has_gdv_layer_data"].sum())
print("Layers where both are present:", ((df_compare["has_gdv_per_layer"]) & (df_compare["has_gdv_layer_data"])).sum())
print("Layers where values match:", df_compare["same_exactish"].sum())

print("\nRows where one side is missing or values differ:")
display(df_compare[
    (~df_compare["has_gdv_per_layer"]) |
    (~df_compare["has_gdv_layer_data"]) |
    (~df_compare["same_exactish"])
])

Layers with gdv_per_layer present: 82
Layers with gdv_layer_data present: 82
Layers where both are present: 82
Layers where values match: 82

Rows where one side is missing or values differ:


,layer_key,gdv_per_layer,gdv_layer_data,same_exactish,difference,has_gdv_per_layer,has_gdv_layer_data


In [8]:
# Cell 1: load the raw activation file

import os
import numpy as np
from pprint import pprint

DATA_PATH = "/absolute/path/to/your/raw_results.npy"

assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"

loaded_results = np.load(DATA_PATH, allow_pickle=True).item()

print("Top-level keys:")
print(list(loaded_results.keys()))

results_list = loaded_results["results"]
print("\nNumber of result entries:", len(results_list))

print("\nKeys in first result:")
print(list(results_list[0].keys()))

NameError: name 'results_list' is not defined